In [50]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from pymatgen.core import Composition, Element

In [51]:
INPUT_CSV = "./data/df_practical.csv"

In [52]:
TOXIC_IMPRACTICAL_ELEMENTS = {
    "As", "Sb", "Pb", "Hg", "Cd", "Tl", "Be",   # toxicity concerns
    "Se", "Te",                                  # already excluded earlier, kept for safety
}

In [53]:
def is_non_toxic(elements_list):
    symbols = [str(el) for el in elements_list]
    return not any(s in TOXIC_IMPRACTICAL_ELEMENTS for s in symbols)

In [54]:
def build_features(df):
    """Same feature set as train_xgboost_shap.py -- kept in sync intentionally."""
    feat = pd.DataFrame(index=df.index)
    feat["density"] = df["density"]
    feat["density_atomic"] = df["density_atomic"]
    feat["avg_atomic_weight"] = df["avg_atomic_weight"]
    feat["nelements"] = df["nelements"]
    feat["nsites"] = df["nsites"]
    feat["volume_per_atom"] = df["volume"] / df["nsites"]
    feat["energy_above_hull"] = df["energy_above_hull"]
    feat["formation_energy_per_atom"] = df["formation_energy_per_atom"]
 
    crystal_dummies = pd.get_dummies(df["crystal_system"], prefix="crystal")
    feat = pd.concat([feat, crystal_dummies], axis=1)
 
    GROUP_VEC = {"Sc": 3, "Ti": 4, "V": 5, "Cr": 6, "Mn": 7, "Fe": 8, "Co": 9, "Ni": 10, "Cu": 11, "Zn": 12}
 
    def valence_electron_count(comp_dict):
        total_amt = sum(comp_dict.values())
        if total_amt == 0:
            return np.nan
        weighted, counted = 0.0, 0.0
        for el_str, amt in comp_dict.items():
            s = str(el_str)
            if s in GROUP_VEC:
                weighted += GROUP_VEC[s] * amt
                counted += amt
        return weighted / counted if counted else 0.0
 
    def transition_metal_fraction(elements_list):
        tm_set = {"Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn"}
        symbols = [str(el) for el in elements_list]
        return sum(1 for s in symbols if s in tm_set) / len(symbols) if symbols else 0.0
 
    def has_oxygen(elements_list):
        return int("O" in [str(el) for el in elements_list])
 
    def dominant_magnetic_element(elements_list):
        mag_set = ["Fe", "Co", "Ni", "Mn", "Cr"]
        symbols = [str(el) for el in elements_list]
        for m in mag_set:
            if m in symbols:
                return m
        return "None"
 
    def composition_stats(comp_dict, prop_name):
        vals, weights = [], []
        for el_str, amt in comp_dict.items():
            try:
                val = getattr(Element(str(el_str)), prop_name)
                if val is not None:
                    vals.append(float(val))
                    weights.append(amt)
            except Exception:
                continue
        if not vals:
            return np.nan, np.nan
        weights, vals = np.array(weights), np.array(vals)
        mean = np.average(vals, weights=weights)
        spread = np.sqrt(np.average((vals - mean) ** 2, weights=weights))
        return mean, spread
 
    ATOMIC_MOMENT_REF = {"Fe": 2.2, "Co": 1.7, "Ni": 0.6, "Mn": 3.0, "Cr": 0.0,
                          "Cu": 0.0, "Zn": 0.0, "V": 0.0, "Ti": 0.0, "Sc": 0.0}
 
    def moment_prior(comp_dict):
        total_amt = sum(comp_dict.values())
        weighted = sum(ATOMIC_MOMENT_REF.get(str(el), 0.0) * amt for el, amt in comp_dict.items())
        return weighted / total_amt if total_amt else np.nan
 
    feat["transition_metal_fraction"] = df["elements"].apply(transition_metal_fraction)
    feat["has_oxygen"] = df["elements"].apply(has_oxygen)
    feat["valence_electron_count"] = df["composition"].apply(valence_electron_count)
 
    en_stats = df["composition"].apply(lambda c: composition_stats(c, "X"))
    feat["electronegativity_mean"] = en_stats.apply(lambda t: t[0])
    feat["electronegativity_spread"] = en_stats.apply(lambda t: t[1])
 
    radius_stats = df["composition"].apply(lambda c: composition_stats(c, "atomic_radius"))
    feat["atomic_radius_mean"] = radius_stats.apply(lambda t: t[0])
    feat["atomic_radius_spread"] = radius_stats.apply(lambda t: t[1])
 
    feat["atomic_moment_prior"] = df["composition"].apply(moment_prior)
    feat["dominant_magnetic_element"] = df["elements"].apply(dominant_magnetic_element)
 
    dom_dummies = pd.get_dummies(feat["dominant_magnetic_element"], prefix="dom_elem")
    feat = pd.concat([feat.drop(columns=["dominant_magnetic_element"]), dom_dummies], axis=1)
 
    return feat

In [55]:
df = pd.read_csv(INPUT_CSV)

In [56]:
if isinstance(df["elements"].iloc[0], str):
    df["elements"] = df["elements"].apply(ast.literal_eval)
if isinstance(df["composition"].iloc[0], str):
    df["composition"] = df["composition"].apply(ast.literal_eval)

--- Hard toxicity filter (applied BEFORE any scoring, not weighted) ---

In [57]:
df = df[df["elements"].apply(is_non_toxic)].copy()
print(f"Remaining after toxicity exclusion: {len(df)}")

Remaining after toxicity exclusion: 2268


--- Build features and train model with out-of-fold predictions ---

In [58]:
X = build_features(df)
y = df["magnetic_performance_index"]
valid_mask = X.notna().all(axis=1) & y.notna()
X, y, df = X[valid_mask], y[valid_mask], df[valid_mask]

In [59]:
y_log = np.log1p(y)
groups = df["formula"]

In [60]:
gkf = GroupKFold(n_splits=5)
oof_preds = np.zeros(len(df))

Fitting 5 folds for each of 60 candidates, totalling 300 fits

Best CV R^2 (log target): 0.8320
Best params:
{'subsample': 0.8,\
 'reg_lambda': 1.5,\
 'reg_alpha': 0,\
 'n_estimators': 800,\
 'min_child_weight': 1,\
 'max_depth': 4,\
 'learning_rate': 0.1,\
 'colsample_bytree': 0.8}

Load the already-trained, tuned model instead of retraining here

In [61]:
model = xgb.XGBRegressor()
model.load_model("./models/magnet_iq_xgb_model.json")
df["model_predicted_performance"] = np.expm1(model.predict(X))

In [62]:
df["model_agreement"] = 1 - (
    np.abs(df["magnetic_performance_index"] - df["model_predicted_performance"])
    / df["magnetic_performance_index"].clip(lower=0.01)
).clip(0, 1)

In [63]:
df_unique = df.sort_values("magnetic_performance_index", ascending=False).drop_duplicates(
    subset="formula", keep="first"
).copy()

In [64]:
def normalize(series, higher_is_better=True):
    s = series.copy()
    if higher_is_better:
        return (s - s.min()) / (s.max() - s.min())
    return (s.max() - s) / (s.max() - s.min())

In [65]:
df_unique["score_performance"] = normalize(df_unique["magnetic_performance_index"])
df_unique["score_stability"] = normalize(df_unique["energy_above_hull"], higher_is_better=False)
df_unique["score_is_stable_bonus"] = df_unique["is_stable"].astype(float)
df_unique["score_simplicity"] = normalize(df_unique["nelements"], higher_is_better=False)
df_unique["score_model_agreement"] = normalize(df_unique["model_agreement"])

In [66]:
WEIGHT_SCHEMES = {
    "performance_focused": {"score_performance": 0.50, "score_stability": 0.20,
                             "score_is_stable_bonus": 0.15, "score_simplicity": 0.05,
                             "score_model_agreement": 0.10},
    "stability_focused": {"score_performance": 0.25, "score_stability": 0.35,
                           "score_is_stable_bonus": 0.25, "score_simplicity": 0.05,
                           "score_model_agreement": 0.10},
    "balanced": {"score_performance": 0.35, "score_stability": 0.25,
                 "score_is_stable_bonus": 0.20, "score_simplicity": 0.05,
                 "score_model_agreement": 0.15},
}

In [67]:
rank_positions = {}
for scheme_name, weights in WEIGHT_SCHEMES.items():
    composite = sum(df_unique[col] * w for col, w in weights.items())
    ranked = df_unique.assign(composite=composite).sort_values("composite", ascending=False)
    rank_positions[scheme_name] = ranked["formula"].reset_index(drop=True)

In [68]:
top_candidates = pd.concat(
    [rank_positions[s].head(10).rename(s) for s in WEIGHT_SCHEMES], axis=1
)
print("\nTop 10 formula under each weight scheme (columns = scheme):")
print(top_candidates.to_string())


Top 10 formula under each weight scheme (columns = scheme):
  performance_focused stability_focused    balanced
0           Ca(MnGe)2         Ca(MnGe)2   Ca(MnGe)2
1              MgMnGe            MgMnGe      MgMnGe
2               Mn2O3             Mn2O3       Mn2O3
3             Mn2GeO4           Mn2GeO4     Mn2GeO4
4           Mn12Ge4N3          Mn7SiO12    Mn7SiO12
5             Mn2VHO5          Mn7GeO12     Mn2VHO5
6            Mn7SiO12           Mn2VHO5    Mn7GeO12
7           Mn3(BO3)2         Mn3(BO3)2   Mn3(BO3)2
8            Mn7GeO12          Ba2Fe2O5    Ba2Fe2O5
9            Ba2Fe2O5        Fe2(MoO4)3  Fe2(MoO4)3


In [69]:
sets = [set(rank_positions[s].head(10)) for s in WEIGHT_SCHEMES]
robust_top = set.intersection(*sets)
print(f"\nFormulas in top 10 under EVERY weight scheme (robust picks):")
robust_top


Formulas in top 10 under EVERY weight scheme (robust picks):


{'Ba2Fe2O5',
 'Ca(MnGe)2',
 'MgMnGe',
 'Mn2GeO4',
 'Mn2O3',
 'Mn2VHO5',
 'Mn3(BO3)2',
 'Mn7GeO12',
 'Mn7SiO12'}

In [70]:
df_unique.to_csv("./outputs/final_candidate_analysis.csv", index=False)
print("\nSaved full analysis to final_candidate_analysis.csv")


Saved full analysis to final_candidate_analysis.csv
